# Verificación de la capa Bronze — Iceberg

Revisamos las tablas Iceberg en `nessie.bronze` tras la ingesta:
referencias Nessie, metadatos, snapshots, particionado y calidad de datos.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("verify-bronze-iceberg")
spark.version

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 08:53:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/23 08:53:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/23 08:53:45 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to events/app-20260323085343-0004.inprogress. This is unsupported


'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

La ingesta bronze usa WAP: cada mes crea una rama efímera, escribe, valida y
hace merge a `bronze`. Tras el último mes de cada año se crea un tag de auditoría.

In [2]:
# Referencias Nessie (ramas + tags)
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

Nessie references:

  * Branch bronze                                   cf2b528dcaa7
  * Branch main                                     2e1cfa82b035
  * Branch silver                                   2e1cfa82b035
    Tag    bronze-taxi_zone_lookup-2026             cf2b528dcaa7
    Tag    bronze-yellow_taxi-2019                  37aeef5eaadc
    Tag    bronze-yellow_taxi-2020                  40916728e7df
    Tag    bronze-yellow_taxi-2022                  a5f74efa569f
    Tag    bronze-yellow_taxi-2023                  86260d2db7d6
    Tag    bronze-yellow_taxi-2025                  38eac64ade0e



  Branches: 3  |  Tags: 6


In [3]:
# Switch to bronze branch for the rest of the notebook
nessie.set_spark_branch(spark, "bronze")

# Available namespaces
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

+---------+
|namespace|
+---------+
|bronze   |
+---------+



In [4]:
# Tables in the bronze namespace
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|bronze   |taxi_zone_lookup|false      |
|bronze   |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [5]:
# Esquema, particionado, propiedades
spark.sql("DESCRIBE EXTENDED nessie.bronze.yellow_taxi").show(250, truncate=False)

+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                |comment|
+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|VendorID                     |bigint                                             

In [6]:
# Propiedades Iceberg (format version, gc, etc.)
spark.sql("SHOW TBLPROPERTIES nessie.bronze.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |660794923121098513                                              |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |cf2b528dcaa74519a738e0e3f18a2a724345149e66bf3f0e3bec8629ce801d21|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [7]:
# Cada write/overwrite crea un snapshot
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                        |summary                                                 

In [8]:
# History (time-ordered snapshot log)
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-03-23 08:35:10.309|2446029578185904954|NULL               |true               |
|2026-03-23 08:35:41.859|3543775155235839539|2446029578185904954|true               |
|2026-03-23 08:36:17.082|4242473783747503839|3543775155235839539|true               |
|2026-03-23 08:36:44.097|446924134455197753 |4242473783747503839|true               |
|2026-03-23 08:37:11.02 |5601934046872077933|446924134455197753 |true               |
|2026-03-23 08:37:27.717|5673233999209721230|5601934046872077933|true               |
|2026-03-23 08:37:41.603|6996139148782402076|5673233999209721230|true               |
|2026-03-23 08:37:58.7  |7456453799009638166|6996139148782402076|true               |
|2026-03-23 08:38:15.843|1281242953035749465|745645379

## 3b. Crecimiento de registros por tag

Cada tag Nessie captura el estado de `bronze` tras ingestar todos los meses
de un año. Leyendo la tabla en cada tag vemos cómo crece año a año.

In [9]:
# Conteo de registros en cada tag anual para ver el crecimiento incremental
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

tag_rows_df = refs_df.filter(refs_df[type_col].ilike("tag")) \
    .filter(refs_df["name"].rlike(r"^bronze-yellow_taxi-\d{4}$"))
tag_names = [r["name"] for r in tag_rows_df.collect()]

tag_pattern = re.compile(r"bronze-yellow_taxi-(\d{4})")
tag_rows = []
for name in tag_names:
    m = tag_pattern.match(name)
    if m:
        tag_rows.append((name, int(m.group(1))))
tag_rows.sort(key=lambda t: t[1])

growth = []
for tag_name, year in tag_rows:
    nessie.set_spark_branch(spark, tag_name)
    count = spark.table("nessie.bronze.yellow_taxi").count()
    growth.append((tag_name, year, count))

nessie.set_spark_branch(spark, "bronze")

growth_df = spark.createDataFrame(
    [(tag, year, total, total - (growth[i-1][2] if i > 0 else 0))
     for i, (tag, year, total) in enumerate(growth)],
    ["tag", "year", "cumulative_records", "records_added"]
)
growth_df.show(truncate=False)

+-----------------------+----+------------------+-------------+
|tag                    |year|cumulative_records|records_added|
+-----------------------+----+------------------+-------------+
|bronze-yellow_taxi-2019|2019|22612607          |22612607     |
|bronze-yellow_taxi-2020|2020|38324669          |15712062     |
|bronze-yellow_taxi-2022|2022|47395913          |9071244      |
|bronze-yellow_taxi-2023|2023|56780400          |9384487      |
|bronze-yellow_taxi-2025|2025|67978426          |11198026     |
+-----------------------+----+------------------+-------------+



## 4. Particionado y ficheros

In [10]:
# Partition spec
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.partitions").show(truncate=False)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{624}    |0      |2463900     |2         |38154028                     |0                           |0                         |0                           |0                         |2026-03-23 08:37:58.7  |7456453799009638166     |
|{625}    |0      |2979419     |3         |45723357         

In [11]:
# Ficheros de datos: rutas, tamaños, registros por fichero
files_df = spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format",
    "record_count", "file_size_in_bytes"
).show(100, truncate=80)

+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/bronze/yellow_taxi_a7166143-45d5-4b7d-b3f7-a94f0fc81f7d/data/...|    PARQUET|     4145224|          65386491|
|      0|s3a://warehouse/bronze/yellow_taxi_a7166143-45d5-4b7d-b3f7-a94f0fc81f7d/data/...|    PARQUET|           1|              5908|
|      0|s3a://warehouse/bronze/yellow_taxi_a7166143-45d5-4b7d-b3f7-a94f0fc81f7d/data/...|    PARQUET|           2|              6376|
|      0|s3a://warehouse/bronze/yellow_taxi_a7166143-45d5-4b7d-b3f7-a94f0fc81f7d/data/...|    PARQUET|           1|              5908|
|      0|s3a://warehouse/bronze/yellow_taxi_a7166143-45

In [12]:
# Resumen: ficheros totales, tamaño, registros
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

+-----------+-------------+-------------+
|total_files|total_size_mb|total_records|
+-----------+-------------+-------------+
|        122|        967.8|     67978426|
+-----------+-------------+-------------+



## 5. Manifiestos

In [13]:
# Una entrada por snapshot
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.manifests").show(truncate=80)

+-------+--------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------+
|content|                                                                            path|length|partition_spec_id|  added_snapshot_id|added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|               partition_summaries|
+-------+--------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------+
|      0|s3a://warehouse/

## 6. Calidad de datos

In [14]:
df = spark.table("nessie.bronze.yellow_taxi")
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Total records: 67,978,426
Columns: 20
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [15]:
# Registros por año-mes (valida el particionado)
from pyspark.sql.functions import year, month

df.groupBy(
    year("tpep_pickup_datetime").alias("year"),
    month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2001|    1|      3|
|2001|    2|      1|
|2002|   12|      3|
|2003|    1|      8|
|2007|   12|      1|
|2008|   12|     92|
|2009|    1|    227|
|2012|    2|      1|
|2014|   11|      1|
|2018|   11|     11|
|2018|   12|    355|
|2019|    1|7696390|
|2019|    2|7049178|
|2019|    3|7866239|
|2019|    4|    114|
|2019|    5|     48|
|2019|    6|     29|
|2019|    7|     21|
|2019|    8|      7|
|2019|    9|      2|
|2019|   12|    131|
|2020|    1|6405096|
|2020|    2|6299397|
|2020|    3|3007294|
|2020|    4|     18|
|2020|    5|     16|
|2020|    6|     16|
|2020|    7|     10|
|2020|    8|      3|
|2021|    1|      3|
|2021|   12|     24|
|2022|    1|2463900|
|2022|    2|2979419|
|2022|    3|3627815|
|2022|    4|     34|
|2022|    5|     19|
|2022|   10|     11|
|2022|   12|     25|
|2023|    1|3066726|
|2023|    2|2914003|
|2023|    3|3403619|
|2023|    4|     85|
|2024|   12|     21|
|2025|    1|3475234|
|2025|    2|3

In [16]:
# Nulos por columna
from pyspark.sql.functions import col as c, sum as spark_sum

null_counts = df.select([
    spark_sum(c(col).isNull().cast("int")).alias(col)
    for col in df.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 3034875  
 trip_distance         | 0        
 RatecodeID            | 3034875  
 store_and_fwd_flag    | 3034875  
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 7891090  
 airport_fee           | 41115652 
 cbd_congestion_fee    | 56780400 



In [17]:
# Estadísticas básicas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "passenger_count"
).summary("count", "min", "max", "mean").show()

+-------+------------------+-----------------+----------------+----------------+------------------+
|summary|     trip_distance|      fare_amount|      tip_amount|    total_amount|   passenger_count|
+-------+------------------+-----------------+----------------+----------------+------------------+
|  count|          67978426|         67978426|        67978426|        67978426|          64943551|
|    min|            -30.62|          -1807.6|         -493.22|        -1832.85|               0.0|
|    max|         348798.53|         943274.8|       141492.02|      1084772.17|               9.0|
|   mean|3.9636783755485023|14.40493741176365|2.47229566819092|20.9186918282815|1.4608446033386748|
+-------+------------------+-----------------+----------------+----------------+------------------+



In [18]:
# Sample rows
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2023-03-01 00:06:43 |2023-03-01 00:16:43  |1.0            |0.0          |1.0       |N                 |238         |42  

## 7. Taxi Zone Lookup

In [23]:
# Check if the reference table exists and inspect it
try:
    zones = spark.table("nessie.bronze.taxi_zone_lookup")
    print(f"Records: {zones.count()}")
    zones.printSchema()
    zones.show(10, truncate=False)
except Exception as e:
    print(f"Table not yet ingested: {e}")

Table not yet ingested: An error occurred while calling o317.count.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConn